In [1]:
import sqlite3
import pandas as pd


PATTERNS_DB_PATH = r"/workspace/qgi\trojan 2.0\data\database\patterns_database.db"


def sqlite_execute_query(query_params: tuple):
    conn = sqlite3.connect(PATTERNS_DB_PATH)
    cursor = conn.cursor()
    query, params = query_params
    cursor.execute(query, params)
    # Check if it's a modification query and commit if it is
    if query.strip().upper().startswith(("INSERT", "UPDATE", "DELETE")):
        conn.commit()
        result = cursor.rowcount  # This will return the number of rows affected
    else:
        result = cursor.fetchall()  # For SELECT statements, fetch the results
    conn.close()
    return result

def migrate_patternized_countries(table_name):
    # Step 1: Rename the existing table
    rename_query = f"ALTER TABLE {table_name} RENAME TO temp_{table_name}"
    sqlite_execute_query((rename_query, ()))

    # Step 2: Create the new table with the updated primary key structure
    create_new_table_query = f'''
    CREATE TABLE IF NOT EXISTS {table_name} (
        org TEXT,
        version INTEGER,
        main_name TEXT,
        index_name TEXT,
        country_name TEXT,
        processing_duration TEXT,
        PRIMARY KEY (org, version, main_name, index_name, country_name)
    )
    '''
    sqlite_execute_query((create_new_table_query, ()))

    # Step 3: Copy records from the old table to the new table
    # Assuming that the 'version' can be set to a default value (e.g., 1) for existing records
    # If 'version' needs to be retrieved from another source, you will need to adjust the query accordingly
    copy_records_query = f'''
    INSERT INTO {table_name} (org, version, main_name, index_name, country_name, processing_duration)
    SELECT org, 1 as version, main_name, index_name, country_name, processing_duration FROM temp_{table_name}
    '''
    sqlite_execute_query((copy_records_query, ()))

    # Step 4: Drop the old table
    drop_old_table_query = f"DROP TABLE temp_{table_name}"
    sqlite_execute_query((drop_old_table_query, ()))

In [2]:
create_table_query = '''
    CREATE TABLE IF NOT EXISTS disqualified_indexes (
        org TEXT,
        version INTEGER,
        main_name TEXT,
        index_name TEXT,
        reason TEXT,
        PRIMARY KEY (org, version, main_name, index_name)
    )
    '''
    
sqlite_execute_query((create_table_query, ()))

In [4]:
conn = sqlite3.connect(PATTERNS_DB_PATH)
df = pd.read_sql_query("SELECT * FROM patterns", conn)
nb = pd.read_sql_query("SELECT COUNT(*) FROM patterns", conn).iloc[0,0]
#conn.close()

print(nb)
df.head()

3030240


,org,version,main_name,index_name,country_a,country_b,pattern_length,correlation,start_year_a,end_year_a,index_values_a,start_year_b,end_year_b,index_values_b,creation_date,matrix_profile,unique_id
0,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Sudan,Libya,3,0.846154,2009,2011,"112.4,111.8,108.7",2008,2010,"70.0,69.4,69.1",2023-11-05 21:28:36,"[0.05261336557346185 15 -1 15],[0.399619100669...",Sudan_Libya_3_Fragile State Index_GS4SB4BN
1,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Sudan,Libya,3,0.958343,2009,2011,"112.4,111.8,108.7",2009,2011,"69.4,69.1,68.7",2023-11-05 21:28:36,"[0.05261336557346185 15 -1 15],[0.399619100669...",Sudan_Libya_3_Fragile State Index_BV6DODYQ
2,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Sudan,Libya,3,0.994781,2009,2011,"112.4,111.8,108.7",2016,2018,"96.4,96.3,94.6",2023-11-05 21:28:36,"[0.05261336557346185 15 -1 15],[0.399619100669...",Sudan_Libya_3_Fragile State Index_3IKE6BYR
3,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Sudan,Libya,3,0.962780,2009,2011,"112.4,111.8,108.7",2017,2019,"96.3,94.6,92.2",2023-11-05 21:28:36,"[0.05261336557346185 15 -1 15],[0.399619100669...",Sudan_Libya_3_Fragile State Index_8A7SEZXY
4,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Sudan,Libya,3,0.646378,2009,2011,"112.4,111.8,108.7",2020,2022,"95.19371849236182,97.0,94.29999999999998",2023-11-05 21:28:36,"[0.05261336557346185 15 -1 15],[0.399619100669...",Sudan_Libya_3_Fragile State Index_XY7PXA90


In [5]:
conn = sqlite3.connect(PATTERNS_DB_PATH)
df = pd.read_sql_query("SELECT * FROM patternized_indexes", conn)
nb = pd.read_sql_query("SELECT COUNT(*) FROM patternized_indexes", conn).iloc[0,0]
#conn.close()

print(nb)
df

1


,org,version,main_name,index_name,processing_duration
0,The Fund For Peace,2023,Fragile State Index,Fragile State Index,3:21:24.416137


In [6]:
conn = sqlite3.connect(PATTERNS_DB_PATH)
df = pd.read_sql_query("SELECT * FROM patternized_countries", conn)
nb = pd.read_sql_query("SELECT COUNT(*) FROM patternized_countries", conn).iloc[0,0]
#conn.close()

print(nb)
df

329


,org,version,main_name,index_name,country_name,processing_duration
0,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Sudan,0:00:41.194449
1,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Libya,0:00:15.106262
2,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Botswana,0:00:33.655204
3,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Jamaica,0:00:32.232108
4,The Fund For Peace,2023,Fragile State Index,Fragile State Index,Malaysia,0:00:37.733360
...,...,...,...,...,...,...
324,The Fund For Peace,2023,Fragile State Index,C1: Security Apparatus,Equatorial Guinea,0:01:27.647622
325,The Fund For Peace,2023,Fragile State Index,C1: Security Apparatus,Antigua and Barbuda,0:00:41.992262
326,The Fund For Peace,2023,Fragile State Index,C1: Security Apparatus,Trinidad and Tobago,0:01:00.432406
327,The Fund For Peace,2023,Fragile State Index,C1: Security Apparatus,Bahamas,0:00:57.427350


In [7]:
conn = sqlite3.connect(PATTERNS_DB_PATH)
df = pd.read_sql_query("SELECT * FROM pattern_summary", conn)
nb = pd.read_sql_query("SELECT COUNT(*) FROM pattern_summary", conn).iloc[0,0]
#conn.close()

print(nb)
df

0


,country_a,country_b,pattern_length,start_year_a,start_year_b,strength,indexes,index_values_a,index_values_b,last_updated
